# Layer 3 Architecture Exploration (FPN vs DeepLabV3Plus vs Unet++)

This notebook compares segmentation architectures while keeping the backbone fixed.

Fixed design:
- Backbone: `resnet34`
- Encoder weights: `imagenet`
- Architectures: `FPN`, `DeepLabV3Plus`, `UnetPlusPlus` (this is the argument name used by `train_ttd.py`)
- Settings: `Single-TB`, `Shift-TA_TC-to-TB_10pct`, `Shift-TA_TB-to-TC_100pct`


## 1. Setup

In [20]:
from pathlib import Path
import subprocess
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path('/users/7/yu001011/csci5527/CSCI5527-final')
SCRIPT_DIR = PROJECT_ROOT / 'baseline_models_scripts'
RESULTS_DIR = SCRIPT_DIR / 'runs'
VENV_PY = Path('/users/7/yu001011/csci5527/.venv/bin/python')

PROJECT_ROOT, SCRIPT_DIR, RESULTS_DIR, VENV_PY

(PosixPath('/users/7/yu001011/csci5527/CSCI5527-final'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts'),
 PosixPath('/users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs'),
 PosixPath('/users/7/yu001011/csci5527/.venv/bin/python'))

## 2. Experiment Design

In [21]:
BACKBONE = 'efficientnet-b0' # resnet34 / efficientnet-b0
ENCODER_WEIGHTS = 'imagenet'
EPOCHS = 100
BATCH_SIZE = 16
IMG_SIZE = 512
LR = 1e-4

SETTINGS = [
    'Single-TB',
    'Shift-TA_TC-to-TB_10pct',
    'Shift-TA_TB-to-TC_10pct',
]

ARCHES = [
    'FPN',
    'DeepLabV3Plus',
    'UnetPlusPlus',
#     'Segformer',
]

pd.DataFrame([
    {'Setting': setting, 'Architecture': arch}
    for setting in SETTINGS
    for arch in ARCHES
])

,Setting,Architecture
0,Single-TB,FPN
1,Single-TB,DeepLabV3Plus
2,Single-TB,UnetPlusPlus
3,Shift-TA_TC-to-TB_10pct,FPN
4,Shift-TA_TC-to-TB_10pct,DeepLabV3Plus
5,Shift-TA_TC-to-TB_10pct,UnetPlusPlus
6,Shift-TA_TB-to-TC_10pct,FPN
7,Shift-TA_TB-to-TC_10pct,DeepLabV3Plus
8,Shift-TA_TB-to-TC_10pct,UnetPlusPlus


## 3. Helper Functions

In [22]:
def result_filename(arch: str) -> str:
    safe_arch = arch.replace('+', 'plus').replace(' ', '_').lower()
    return f'layer3_arch_{safe_arch}_{BACKBONE}.csv'

def build_command(setting: str, arch: str):
    return [
        str(VENV_PY),
        'train_ttd.py',
        '--project-root', str(PROJECT_ROOT),
        '--experiment', setting,
        '--arch', arch,
        '--encoder', BACKBONE,
        '--encoder-weights', ENCODER_WEIGHTS,
        '--epochs', str(EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--img-size', str(IMG_SIZE),
        '--lr', str(LR),
        '--results-name', result_filename(arch),
    ]

def run_one(setting: str, arch: str):
    cmd = build_command(setting, arch)
    print('Running:', ' '.join(cmd))
    return subprocess.run(cmd, cwd=SCRIPT_DIR, check=True)

def run_all(settings, arches):
    for arch in arches:
        for setting in settings:
            run_one(setting, arch)


## 4. Preview Commands

In [23]:
command_preview = pd.DataFrame([
    {
        'Setting': setting,
        'Architecture': arch,
        'Command': ' '.join(build_command(setting, arch)),
    }
    for setting in SETTINGS
    for arch in ARCHES
])

display(command_preview)

,Setting,Architecture,Command
0,Single-TB,FPN,/users/7/yu001011/csci5527/.venv/bin/python tr...
1,Single-TB,DeepLabV3Plus,/users/7/yu001011/csci5527/.venv/bin/python tr...
2,Single-TB,UnetPlusPlus,/users/7/yu001011/csci5527/.venv/bin/python tr...
3,Shift-TA_TC-to-TB_10pct,FPN,/users/7/yu001011/csci5527/.venv/bin/python tr...
4,Shift-TA_TC-to-TB_10pct,DeepLabV3Plus,/users/7/yu001011/csci5527/.venv/bin/python tr...
5,Shift-TA_TC-to-TB_10pct,UnetPlusPlus,/users/7/yu001011/csci5527/.venv/bin/python tr...
6,Shift-TA_TB-to-TC_10pct,FPN,/users/7/yu001011/csci5527/.venv/bin/python tr...
7,Shift-TA_TB-to-TC_10pct,DeepLabV3Plus,/users/7/yu001011/csci5527/.venv/bin/python tr...
8,Shift-TA_TB-to-TC_10pct,UnetPlusPlus,/users/7/yu001011/csci5527/.venv/bin/python tr...


## 5. Run a Single Experiment

In [24]:
TEST_SETTING = 'Single-TB'
TEST_ARCH = 'Unet'

# Uncomment to run:
# run_one(TEST_SETTING, TEST_ARCH)

## 6. Run the Full Layer 2 Grid

In [25]:
# Uncomment to run all experiments:
run_all(SETTINGS, ARCHES)

Running: /users/7/yu001011/csci5527/.venv/bin/python train_ttd.py --project-root /users/7/yu001011/csci5527/CSCI5527-final --experiment Single-TB --arch FPN --encoder efficientnet-b0 --encoder-weights imagenet --epochs 100 --batch-size 16 --img-size 512 --lr 0.0001 --results-name layer3_arch_fpn_efficientnet-b0.csv
Python executable: /users/7/yu001011/csci5527/.venv/bin/python
Torch version: 2.11.0+cu126
CUDA available: True
CUDA device count: 1
GPU 0: NVIDIA A100-SXM4-40GB
Using device: cuda:0

===== Running Single-TB | arch=FPN | encoder=efficientnet-b0 =====
Loading CSV metadata...
Sanitizing training and validation masks...
Sanitizing test masks...


Sanitizing Masks: 100%|██████████| 48/48 [00:00<00:00, 3274.35it/s]


Generating Dataloaders...

Pipeline Ready:
 - Training batches: 20
 - Validation batches: 6
 - Testing batches: 3
  → FPN-efficientnet-b0-imagenet_Single-TB:  37%|███▋      | 37/100 [06:10<09:47,  9.33s/it, IoU=0.2792, F1=0.4260]Epoch 37: T-Loss: 6.4547 | V-Loss: 6.2733 | IoU: 0.2792 | F1: 0.4260

Early Stopping Triggered at Epoch 37
  - Best Validation Loss: 6.2421
  - Patience Exhausted: 10/10 epochs without improvement
  - Stopping training now.
Charts saved to /users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/figures/                    
Visualization saved to figures/FPN-efficientnet-b0-imagenet_Single-TB_visualization.png
Saved interim results to: /users/7/yu001011/csci5527/CSCI5527-final/baseline_models_scripts/runs/layer3_arch_fpn_efficientnet-b0.csv

Final results:
                               Experiment  Val_Loss  ...  Test_Recall  Test_Prec
0  FPN-efficientnet-b0-imagenet_Single-TB  6.258277  ...     0.388011   0.439543

[1 rows x 11 columns]


Running: /users/7/yu001011/csci5527/.venv/bin/python train_ttd.py --project-root /users/7/yu001011/csci5527/CSCI5527-final --experiment Shift-TA_TC-to-TB_10pct --arch FPN --encoder efficientnet-b0 --encoder-weights imagenet --epochs 100 --batch-size 16 --img-size 512 --lr 0.0001 --results-name layer3_arch_fpn_efficientnet-b0.csv
Python executable: /users/7/yu001011/csci5527/.venv/bin/python
Torch version: 2.11.0+cu126
CUDA available: True
CUDA device count: 1
GPU 0: NVIDIA A100-SXM4-40GB
Using device: cuda:0

===== Running Shift-TA_TC-to-TB_10pct | arch=FPN | encoder=efficientnet-b0 =====
Loading CSV metadata...
Sanitizing training and validation masks...


Sanitizing Masks: 100%|██████████| 48/48 [00:00<00:00, 14191.92it/s]


...

Sanitizing test masks...
Generating Dataloaders...

Pipeline Ready:
 - Training batches: 45
 - Validation batches: 13
 - Testing batches: 3
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TC-to-TB_10pct:   0%|          | 0/100 [00:22<?, ?it/s, IoU=0.1642, F1=0.2737]Epoch 0: T-Loss: 9.3583 | V-Loss: 8.5377 | IoU: 0.1642 | F1: 0.2737 [Saved Best Model]
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TC-to-TB_10pct:   1%|          | 1/100 [00:45<38:07, 23.11s/it, IoU=0.2633, F1=0.4021]Epoch 1: T-Loss: 8.5533 | V-Loss: 8.0154 | IoU: 0.2633 | F1: 0.4021 [Saved Best Model]
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TC-to-TB_10pct:   2%|▏         | 2/100 [01:10<36:46, 22.52s/it, IoU=0.3165, F1=0.4624]Epoch 2: T-Loss: 8.0750 | V-Loss: 7.7781 | IoU: 0.3165 | F1: 0.4624 [Saved Best Model]
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TC-to-TB_10pct:   3%|▎         | 3/100 [01:35<38:43, 23.95s/it, IoU=0.3226, F1=0.4675]Epoch 3: T-Loss: 7.7196 | V-Loss: 7.4121 | IoU: 0.3226 | F1

  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TC-to-TB_10pct:  38%|███▊      | 38/100 [15:19<26:48, 25.94s/it, IoU=0.5085, F1=0.6512]Epoch 38: T-Loss: 5.4081 | V-Loss: 5.6524 | IoU: 0.5085 | F1: 0.6512
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TC-to-TB_10pct:  39%|███▉      | 39/100 [15:44<27:07, 26.67s/it, IoU=0.4985, F1=0.6420]Epoch 39: T-Loss: 5.3783 | V-Loss: 5.6445 | IoU: 0.4985 | F1: 0.6420
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TC-to-TB_10pct:  40%|████      | 40/100 [16:07<26:02, 26.05s/it, IoU=0.5045, F1=0.6485]Epoch 40: T-Loss: 5.3825 | V-Loss: 5.5811 | IoU: 0.5045 | F1: 0.6485
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TC-to-TB_10pct:  41%|████      | 41/100 [16:30<24:47, 25.22s/it, IoU=0.4658, F1=0.6123]Epoch 41: T-Loss: 5.4196 | V-Loss: 5.6569 | IoU: 0.4658 | F1: 0.6123
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TC-to-TB_10pct:  42%|████▏     | 42/100 [16:56<23:52, 24.70s/it, IoU=0.5080, F1=0.6515]Epoch 42: T-Loss: 5.3793 | V-Loss:

Sanitizing Masks: 100%|██████████| 38/38 [00:00<00:00, 2933.13it/s]


Sanitizing test masks...
Generating Dataloaders...

Pipeline Ready:
 - Training batches: 50
 - Validation batches: 15
 - Testing batches: 3
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct:   0%|          | 0/100 [00:43<?, ?it/s, IoU=0.0993, F1=0.1760]Epoch 0: T-Loss: 9.3801 | V-Loss: 8.9251 | IoU: 0.0993 | F1: 0.1760 [Saved Best Model]
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct:   1%|          | 1/100 [01:26<1:11:06, 43.10s/it, IoU=0.1724, F1=0.2843]Epoch 1: T-Loss: 8.5929 | V-Loss: 8.2602 | IoU: 0.1724 | F1: 0.2843 [Saved Best Model]
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct:   2%|▏         | 2/100 [02:09<1:10:44, 43.31s/it, IoU=0.2521, F1=0.3871]Epoch 2: T-Loss: 8.0348 | V-Loss: 7.8329 | IoU: 0.2521 | F1: 0.3871 [Saved Best Model]
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct:   3%|▎         | 3/100 [02:46<1:09:59, 43.29s/it, IoU=0.1752, F1=0.2850]Epoch 3: T-Loss: 7.6110 | V-Loss: 7.5838 | IoU: 0.175

  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct:  37%|███▋      | 37/100 [21:20<34:00, 32.39s/it, IoU=0.5133, F1=0.6640]Epoch 37: T-Loss: 5.4779 | V-Loss: 5.4079 | IoU: 0.5133 | F1: 0.6640
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct:  38%|███▊      | 38/100 [21:53<33:39, 32.57s/it, IoU=0.5337, F1=0.6838]Epoch 38: T-Loss: 5.4864 | V-Loss: 5.3948 | IoU: 0.5337 | F1: 0.6838
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct:  39%|███▉      | 39/100 [22:26<33:20, 32.79s/it, IoU=0.5404, F1=0.6899]Epoch 39: T-Loss: 5.4835 | V-Loss: 5.4203 | IoU: 0.5404 | F1: 0.6899
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct:  40%|████      | 40/100 [23:03<32:44, 32.74s/it, IoU=0.5334, F1=0.6852]Epoch 40: T-Loss: 5.4833 | V-Loss: 5.3914 | IoU: 0.5334 | F1: 0.6852
  → UnetPlusPlus-efficientnet-b0-imagenet_Shift-TA_TB-to-TC_10pct:  41%|████      | 41/100 [23:36<33:29, 34.06s/it, IoU=0.5439, F1=0.6942]Epoch 41: T-Loss: 5.4732 | V-Loss:

## 7. Load Architecture Results

In [26]:
def load_arch_results(arches):
    frames = []
    for arch in arches:
        path = RESULTS_DIR / result_filename(arch)
        if path.exists():
            df = pd.read_csv(path)
            df['Architecture'] = arch
            frames.append(df)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

arch_df = load_arch_results(ARCHES)
arch_df

""


## 8. Pivot Table for Comparison

In [27]:
if not arch_df.empty:
    arch_df = arch_df.copy()
    arch_df['Setting'] = arch_df['Experiment'].str.replace(r'^.+?_.+?_', '', regex=True)
    pivot_iou = arch_df.pivot(index='Setting', columns='Architecture', values='Test_IoU')
    pivot_f1 = arch_df.pivot(index='Setting', columns='Architecture', values='Test_F1')
    display(Markdown('### Test IoU'))
    display(pivot_iou)
    display(Markdown('### Test F1'))
    display(pivot_f1)
else:
    display(Markdown('No results available yet.'))

No results available yet.